In [1]:
#  Install required packages
!pip install -qU langchain langchain-google-genai google-generativeai==0.8.5


In [9]:
import os


# LangChain + Gemini
import langchain_core, langchain

from langchain_google_genai import ChatGoogleGenerativeAI # to work with Geminiai
from langchain_core.output_parsers import StrOutputParser # to extract plain text from responses
from langchain_core.runnables import RunnablePassthrough # chaining steps
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory


In [6]:
#  Set your Gemini API key\n AQ.Ab8RN6JYLjGEeCfbNQP_LxYuD-jYSS9wQirMdTyJnph9ST8HKg
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"
):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")
print(os.environ.get("GOOGLE_API_KEY"))

AQ.Ab8RN6JYLjGEeCfbNQP_LxYuD-jYSS9wQirMdTyJnph9ST8HKg


In [7]:
# Recommended for demos: fast & economical
model_name = "gemini-3.6-flash"

llm = ChatGoogleGenerativeAI(
    model=model_name,
    convert_system_message_to_human=True,  # smoother behavior with system prompts
    temperature=0.7, # to enconrage a bit of creativity in responses 
)
print(f" Ready with model: {model_name}")

 Ready with model: gemini-3.6-flash


User message
     │
     ▼
Message History
     │
     ▼
ChatPromptTemplate
     │
     ▼
Chat Model
     │
     ▼
AI response
     │
     └──────► Message History

In [10]:
# Create a chat prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

In [11]:
# Create the basic chain
chain = prompt | llm

In [12]:
# Store conversation histories
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [13]:
# Add message history to the chain
conversation_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

In [14]:
response = conversation_chain.invoke(
    {"input": "Hi, my name is Alice."},
    config={"configurable": {"session_id": "user_1"}}
)

print(response.content)

c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': "Hello Alice! It's nice to meet you. How can I help you today?", 'extras': {'signature': 'EooHCocHARFNMg8+/FCh17REcbbxeXOIfIwaOG6u0IqeJr4hdpF9mPJTL8feT4E0flNBvaw9Y/vjx4vb12RMvRWh/GBZOe0OnPE2i130E/9veO61+PyzLpYsBGjVPxqeG+FztafUSjIZkjSyE9/Kb62mLiwmHOd6+Y5fcG+MYscBRVKcKCWflfzR9Ps3tHfJkxx1dkUaUWUT+HchU1E/ALW57AUhkwGT/XLqetVHI22qrmC+N3+Z2kC5S5pLwuWu+wlGG+YPNlrOPfso6OzGazvqGh2CzkH5fCBcWSX3jaR6BF0COfTeDZUVsD8udh7cZLRmzU0+FDLtio2oB/fUKGz5JDz6Uefsc79JAnB3Sbpqia+2Wj+fM1uMGMzvMKWvaTCumVUe1gw8jJZp+C6B3f36B8/4b2gtSSC8myzDYCq0g7ybYFNZkll87bxD1Lx4PNGtOQN9ODmxV4X62/FfFApBfP3rRe39vbrn79PZk8WZrAe9vvN33Kpw5OYAQ6R8Mo8lkK+Vjwn/y2UVpW6DlqgGvLGmkJwBS8FfhQhwv8rPwXVb3dVlYy7THH+UFWREc17tOSau+xlTi9GzM6krm53pySf8YCZU3weH2JpxixwLNiHoXDeHFKyg+LDU6ydNSZ5EEWwYKvpGEQM8BysN9yF5YIP6StqY/SEzC1ubj13+Z1A65hkaAI6eu8JE3xMlUfgn3O2hXJAfFq9PXLXjsK5hTDmK20gQEblY4yhw2SkAGcHibG3OZsHj/HCvbp962KflzypuN+huvWglFcpwnfRaughuwy7e/VJnjNLmzwr+yCbv2DpsqG69/z4by8AhiVkXNf80FPRGWLNfEo/Vl/ry7IPGznNkzYYzwiRulIg5c/LsvbP8

In [15]:
response = conversation_chain.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user_1"}}
)

print(response.content)

c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Your name is Alice! How can I help you today, Alice?', 'extras': {'signature': 'EokECoYEARFNMg/myUOgnkrlaS/hxXTS6XnEFPGNufRvnqc9V0pDOmvUz5IvZ8QnOQ/6BHIInGeNJ+iMHsVflPdS7nis2+o7o+ZZrIaCkt262oKsS+KHq6UgaU3dki8GJsy3IsfcZMfzgwg2Dzv24RUanQmLEb5ceg7c2x9/KMcu9cC6kI6cODnY91Yy1ZgVPx5gvly1me7SJMEon6xp/Y+EYsuPuIz+RvxyKgQOKYlZCKe7+kwQ56V5GiMtC++Dpu0wU08wN7r/TgEifNr2cQ76c26xcGyJP5aG3hqaRSYKTzSeDVs/n7woy75qWvUSXxmrQaK5EdpYc3lHiNjrMQftPrXJ7Lt/UZR5rS7BpRGzO01hIt08hSqauDJctsBi0D/uh+eaDcUgdneM0o1wsue1EAYinnwjxvXPviA+/TCDtwTPDhzXmXlD1wQc4f5EIIaevfULVS8tYL63LbghycUyOGP23CC3eE7+oUvSWQD0+HYGPSLQls8OQehOe544nF+lkzQMo2wHNND6FU7iUuIfueKMLDytbLYC9OpM7ioAsYOUdtnzWSqErzWTTFEHcrE2c1N4CHbkE/URwdeRd1JI0wKizrk68dkcN0l/KdA+dsOgjhf1/OohQsPDVP3x0lWr/L9lYjjv0Tc7UTCc4Eh2KS0kV1O0HjcWVphiXFboGNZPhBrzex05l/g='}}]
